# Transformer Decoder

## Code

In [1]:
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn
import models.deep_learning.components as comp


## Testing

In [2]:
# input parameters
N = 3
M = 4
batch_size = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float32

# Decoder Layer parameters
d_model = 4
nhead = 2
dim_feedforward = 64
dropout = 0.2
layer_norm_eps = 1e-5
norm_first = True
bias = True

src_mask = comp.create_random_mask((N, N), device=device)
tgt_mask = comp.create_causal_mask((M, M), device=device)
memory_mask = comp.create_random_mask((M, N), device=device)


## Transformer decoder parameters
num_enc_layers = 3
num_dec_layers = 5
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [3]:
torch.manual_seed(0)
x = torch.randn(batch_size, N, d_model, device=device, dtype=dtype)
tgt = torch.randn(batch_size, M, d_model, device=device, dtype=dtype)

In [4]:
init_seed = 42  # avoide weights initialization randomness effects
train_seed = 24  # avoid dropout randomness effects

In [5]:
torch.manual_seed(init_seed)
tf = mynn.Transformer(
    d_model,
    nhead,
    num_encoder_layers=num_enc_layers,
    num_decoder_layers=num_dec_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation_cls=nn.GELU,
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)

torch.manual_seed(init_seed)

nn_tf = nn.Transformer(
    d_model,
    nhead,
    num_encoder_layers=num_enc_layers,
    num_decoder_layers=num_dec_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation="gelu",
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
    batch_first=True,
)

tf.load_weights_from_torch_transformer(nn_tf)

/Users/abeldiaz/Documents/learn/cs/3-Machine Learning/ml-notebook/.venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


### Evaluation

In [6]:
tf.eval()
tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-0.1642,  0.9794,  0.7503, -1.5655],
         [-0.4250,  1.2774,  0.5289, -1.3813],
         [-1.0446,  1.0612,  0.9358, -0.9523],
         [-1.2869,  1.4140,  0.3467, -0.4737]],

        [[-0.3913,  1.3783, -1.3479,  0.3608],
         [ 0.0958,  0.5395, -1.6401,  1.0048],
         [ 0.3971, -1.7188,  0.7461,  0.5757],
         [ 0.8195, -1.6769,  0.1560,  0.7015]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)

In [7]:
nn_tf.eval()
nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-0.1642,  0.9794,  0.7503, -1.5655],
         [-0.4250,  1.2774,  0.5289, -1.3813],
         [-1.0446,  1.0612,  0.9358, -0.9523],
         [-1.2869,  1.4140,  0.3467, -0.4737]],

        [[-0.3913,  1.3783, -1.3479,  0.3608],
         [ 0.0958,  0.5395, -1.6401,  1.0048],
         [ 0.3971, -1.7188,  0.7461,  0.5757],
         [ 0.8195, -1.6769,  0.1560,  0.7015]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)

### Training

In [8]:
mse = torch.nn.MSELoss()

In [9]:
torch.manual_seed(train_seed)
nn_tf.train()
out = nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)
print(out)
loss = mse(out, tgt)
loss.backward()
optimizer = torch.optim.SGD(nn_tf.parameters(), lr=1e-3)
optimizer.step()
nn_tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-1.4346e-03,  1.4953e+00, -1.7754e-01, -1.3163e+00],
         [-7.8236e-01,  1.6687e+00, -1.1888e-01, -7.6751e-01],
         [-7.7306e-01,  5.9940e-01,  1.3173e+00, -1.1436e+00],
         [-1.2037e+00,  9.0813e-01,  1.0650e+00, -7.6943e-01]],

        [[-1.1991e-01,  1.2835e+00, -1.4932e+00,  3.2966e-01],
         [-2.7357e-01,  1.2387e+00, -1.4637e+00,  4.9856e-01],
         [-6.6840e-01, -1.1637e+00,  1.4264e+00,  4.0566e-01],
         [ 1.5971e+00, -7.8339e-01,  9.5427e-02, -9.0912e-01]]],
       device='mps:0', grad_fn=<NativeLayerNormBackward0>)


tensor([[[ 1.1610, -0.1108,  0.4959, -1.5464],
         [ 1.2565, -1.3045,  0.6225, -0.5743],
         [ 0.6491, -1.4780, -0.3096,  1.1388],
         [-1.4454,  1.1459, -0.3750,  0.6736]],

        [[ 0.1136,  1.5888, -0.7966, -0.9067],
         [ 1.5856,  0.0064, -1.1249, -0.4672],
         [-0.1144, -1.5218,  1.2242,  0.4121],
         [ 1.1544, -1.5778,  0.4191,  0.0046]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)

In [10]:
torch.manual_seed(train_seed)
tf.train()
out = tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)
print(out)
loss = mse(out, tgt)
loss.backward()
optimizer = torch.optim.SGD(tf.parameters(), lr=1e-3)
optimizer.step()
tf(x, tgt=tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask)

tensor([[[-1.4346e-03,  1.4953e+00, -1.7754e-01, -1.3163e+00],
         [-7.8236e-01,  1.6687e+00, -1.1888e-01, -7.6751e-01],
         [-7.7306e-01,  5.9940e-01,  1.3173e+00, -1.1436e+00],
         [-1.2037e+00,  9.0813e-01,  1.0650e+00, -7.6943e-01]],

        [[-1.1991e-01,  1.2835e+00, -1.4932e+00,  3.2966e-01],
         [-2.7357e-01,  1.2387e+00, -1.4637e+00,  4.9856e-01],
         [-6.6840e-01, -1.1637e+00,  1.4264e+00,  4.0566e-01],
         [ 1.5971e+00, -7.8339e-01,  9.5427e-02, -9.0912e-01]]],
       device='mps:0', grad_fn=<NativeLayerNormBackward0>)


tensor([[[ 1.1598, -0.1068,  0.4947, -1.5480],
         [ 1.2563, -1.2995,  0.6258, -0.5824],
         [ 0.6508, -1.4785, -0.3093,  1.1373],
         [-1.4458,  1.1456, -0.3745,  0.6737]],

        [[ 0.1138,  1.5887, -0.7969, -0.9066],
         [ 1.5858,  0.0058, -1.1249, -0.4668],
         [-0.1137, -1.5225,  1.2232,  0.4131],
         [ 1.1548, -1.5778,  0.4182,  0.0051]]], device='mps:0',
       grad_fn=<NativeLayerNormBackward0>)